# 03 · Sri Lankan dataset pipeline (Colab runner)

Runs the code from the `Data` branch on Colab. Code is tested locally first; this notebook does the full runs.

**Before running**
- Runtime: **CPU is enough** (all models are called through APIs).
- Colab Secrets (key icon, left bar): add `GEMINI_API_KEY`, `OPENROUTER_API_KEY`, `DEEPSEEK_API_KEY`, `HF_TOKEN` and turn on *Notebook access* for each.
- Accept the dataset terms on Hugging Face for gated sources (e.g. `openlanguagedata/flores_plus`).

Outputs go to `MyDrive/PII_Project`. The API cache lives on local disk and is copied to Drive every 5 minutes, so a disconnected session can simply be re-run: answered calls are not paid for twice.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
REPO_URL = "https://github.com/sakuna47/real-time-multilingual-pii-detection.git"
BRANCH = "Data"
REPO = "/content/repo"
if not os.path.exists(REPO):
    !git clone -q -b {BRANCH} {REPO_URL} {REPO}
else:
    !git -C {REPO} fetch -q && git -C {REPO} checkout -q {BRANCH} && git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
import os, sys
os.environ["PII_DATA_ROOT"] = "/content/drive/MyDrive/PII_Project"
sys.path.insert(0, REPO)

from src.common import colab
colab.setup_env()                      # API cache + Hugging Face downloads on local disk
print("secrets found:", colab.load_secrets())
print("cache restored from Drive:", colab.restore_cache())
sync = colab.start_cache_sync(every_min=5)

## 1 · Check providers

In [ ]:
!python -m src.llm.smoke

## 2 · Translator benchmark (FLORES-200 devtest, en → si / ta)

In [ ]:
!python -m src.sources.flores

In [ ]:
# Edit the model list to what your keys can use (python -m src.llm.smoke --list-models).
MODELS = "deepseek:deepseek-flash,deepseek:deepseek-v4-pro,gemini:gemini-2.5-flash"
!python -m src.translate.benchmark --models {MODELS} --langs si,ta

## Finish: save the cache to Drive

In [ ]:
print("saved:", colab.save_cache())
sync.set()